# 01 — Exploratory Data Analysis

This notebook checks the supplied data before training. Run from the notebook directory; paths use `../data/raw/student_performance.csv`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
DATA_PATH = Path("../data/raw/student_performance.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "FinalGrade"
FEATURES = [c for c in df.columns if c != TARGET]
print("Shape:", df.shape)
display(df.head())

## Shape, dtypes, missing values, and duplicates


In [ ]:
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Rows after duplicate removal:", len(df.drop_duplicates()))

## Grade class balance

The target is fairly balanced across four classes; no log transform is needed. `0=High`, `1=Good`, `2=Average`, `3=Low`.


In [ ]:
grade_names = {0:"High", 1:"Good", 2:"Average", 3:"Low"}
counts = df[TARGET].value_counts().sort_index()
display(pd.DataFrame({"count":counts, "category":counts.index.map(grade_names)}))
ax = counts.plot(kind="bar", rot=0, title="FinalGrade class balance")
ax.set_xlabel("FinalGrade code"); ax.set_ylabel("Rows"); plt.show()

## Grade versus ExamScore: target leakage check

**Leakage note:** `FinalGrade` is a direct binning of `ExamScore`. High metrics with ExamScore included are therefore leakage-inflated.


In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x="FinalGrade", y="ExamScore")
plt.title("ExamScore by FinalGrade (direct target relationship)"); plt.show()
print("Observed min/max ExamScore per grade:")
display(df.groupby(TARGET)["ExamScore"].agg(["min","max","count"]))
# Target rules supplied for this dataset: 85–100 -> 0, 70–84 -> 1, 55–69 -> 2, 40–54 -> 3.

## Grade by study habits and context


In [ ]:
for col in ["StudyHours", "Attendance", "Motivation", "StressLevel"]:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=TARGET, y=col)
    plt.title(f"{col} by FinalGrade"); plt.show()

## Correlation heatmap

Because lower grade codes mean better grades, ExamScore and FinalGrade should have a strong negative correlation (approximately −0.97 in the supplied data).


In [ ]:
plt.figure(figsize=(12,9))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Feature correlation heatmap"); plt.show()
print("ExamScore / FinalGrade correlation:", df["ExamScore"].corr(df[TARGET]))

## Basic outlier check

The IQR rule is a flagging tool, not proof that a value is erroneous.


In [ ]:
numeric = ["StudyHours", "Attendance", "Age", "OnlineCourses", "AssignmentCompletion", "ExamScore"]
outlier_rows=[]
for col in numeric:
    q1,q3=df[col].quantile([.25,.75]); iqr=q3-q1
    low,high=q1-1.5*iqr,q3+1.5*iqr
    outlier_rows.append({"column":col,"lower_fence":low,"upper_fence":high,"outlier_count":int(((df[col]<low)|(df[col]>high)).sum())})
display(pd.DataFrame(outlier_rows))